In [3]:
pip install mermaid_diagram

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement mermaid_diagram (from versions: none)
ERROR: No matching distribution found for mermaid_diagram


In [2]:
from mermaid_diagram import MermaidDiagram

ModuleNotFoundError: No module named 'mermaid_diagram'

In [1]:
graph LR
A[Load Data] --> B{Preprocess Data}
B --> C{Create SVM Dataset}
C --> D{Scale Data}
D --> E{Split Train/Test}
E --> F{Train/Evaluate SVM}
F --> G{Choose Best SVM}
G --> H{Create Hybrid Dataset}
H --> I{LSTM/GRU Input}
I --> J{Build/Train LSTM/GRU}
J --> K{Evaluate & Tune Hyperparameters}
K --> L{Choose Best LSTM/GRU}
L --> M{Predict with Best Model}
M --> N{Invert Predictions}
N --> O{Plot Actual vs. Predicted}

SyntaxError: invalid syntax (3486001785.py, line 1)

In [12]:
\documentclass{article}
\usepackage{tikz}
\usetikzlibrary{shapes.geometric, arrows, positioning}

% Define styles
\tikzstyle{block} = [rectangle, draw, rounded corners, minimum height=1.5em, minimum width=7em, text centered, draw=black, fill=blue!10, font=\footnotesize]
\tikzstyle{input} = [rectangle, draw, minimum height=1.5em, minimum width=7em, text centered, draw=black, fill=green!20, font=\footnotesize]
\tikzstyle{output} = [rectangle, draw, minimum height=1.5em, minimum width=7em, text centered, draw=black, fill=red!20, font=\footnotesize]
\tikzstyle{arrow} = [thick,->,>=stealth]

\begin{document}

\begin{tikzpicture}[node distance=1.2cm, scale=0.8, every node/.style={transform shape}]

% Input
\node (input) [input] {Input};

% LSTM Branch
\node (lstm) [block, below left=1.5cm and 0.8cm of input] {LSTM (128 units)};
\node (dropout1) [block, below of=lstm] {Dropout (0.2)};

% GRU Branch
\node (gru) [block, below right=1.5cm and 0.8cm of input] {GRU (64 units)};
\node (dropout2) [block, below of=gru] {Dropout (0.2)};

% Attention Mechanism (centered between LSTM and GRU branches)
\node (attention) [block, below=1.5cm of dropout1, xshift=3.4cm] {Attention Mechanism};

% Dense Layer
\node (dense) [block, below of=attention] {Dense (1 unit)};

% Output
\node (output) [output, below of=dense] {Output};

% Arrows
\draw [arrow] (input.south) -- ++(0,-0.6cm) -| (lstm.north);
\draw [arrow] (input.south) -- ++(0,-0.6cm) -| (gru.north);

\draw [arrow] (lstm) -- (dropout1);
\draw [arrow] (gru) -- (dropout2);

% Arrows to Attention Mechanism
\draw [arrow] (dropout1.south) -- ++(0,-0.6cm) -| ([yshift=0cm]attention.west);
\draw [arrow] (dropout2.south) -- ++(0,-0.6cm) -| ([yshift=0cm]attention.east);

\draw [arrow] (attention) -- (dense);
\draw [arrow] (dense) -- (output);

\end{tikzpicture}

\end{document}


In [13]:
# Create a 'Site' column with the first 7 letters of the 'ERBS' column
data['Site'] = data['ERBS'].str[:7]

# Group by 'Site' and 'STARTTIME_DATE', then average the values
grouped_data = data.groupby(['Site', 'STARTTIME_DATE']).mean().reset_index()

C:\Users\HP\AppData\Local\Temp\ipykernel_115676\202914169.py:5: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  grouped_data = data.groupby(['Site', 'STARTTIME_DATE']).mean().reset_index()


In [14]:
# Normalize the target columns between 0 and 10
scaler = MinMaxScaler(feature_range=(0, 10))
columns_to_normalize = ['AVG_USR_THRPUT_DL', 'AVG_NO_USER', 'DL_TRAFFIC_MB']
grouped_data[columns_to_normalize] = scaler.fit_transform(grouped_data[columns_to_normalize])


In [15]:
# Function to create sequences for LSTM
def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        x = data[i:i+seq_length]
        y = data[i+seq_length]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

In [7]:
# Prediction for a single target
def lstm_prediction(target_column):
    results = {}
    for site in grouped_data['Site'].unique():
        site_data = grouped_data[grouped_data['Site'] == site][target_column].values
        site_data = site_data.reshape(-1, 1)
        
        seq_length = 10
        X, y = create_sequences(site_data, seq_length)
        
        # Split data into train, test, validation
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
        
        # LSTM model
        model = Sequential()
        model.add(LSTM(50, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
        model.add(Dropout(0.2))
        model.add(LSTM(50, return_sequences=False))
        model.add(Dropout(0.2))
        model.add(Dense(1))
        
        model.compile(optimizer='adam', loss='mean_squared_error')
        model.fit(X_train, y_train, epochs=10, batch_size=16, validation_data=(X_val, y_val), verbose=0)
        
        # Predictions and MSE calculation
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)
        val_pred = model.predict(X_val)
        
        train_mse = mean_squared_error(y_train, train_pred)
        test_mse = mean_squared_error(y_test, test_pred)
        val_mse = mean_squared_error(y_val, val_pred)
        
        results[site] = (train_mse, test_mse, val_mse)
    
    return results

In [8]:
# Define the target column for DL_TRAFFIC_MB
target_column = 'DL_TRAFFIC_MB'

# Initialize the dictionary to store predictions
site_predictions_dl_traffic = {}

# Train the model for each site
for site in best_5_sites + worst_5_sites:
    site_data = grouped_data[grouped_data['Site'] == site[0]][target_column].values
    site_data = site_data.reshape(-1, 1)

    seq_length = 10
    if len(site_data) <= seq_length:
        continue
    
    X, y = create_sequences(site_data, seq_length)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Define the LSTM model
    model = Sequential()
    model.add(LSTM(50, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
    model.add(Dropout(0.2))
    model.add(LSTM(50, return_sequences=False))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    
    model.compile(optimizer='adam', loss='mean_squared_error')
    model.fit(X_train, y_train, epochs=10, batch_size=16, validation_split=0.2, verbose=0)
    
    # Store the predictions for plotting and analysis
    y_pred = model.predict(X_test)
    site_predictions_dl_traffic[site[0]] = (X_test, y_test, y_pred)

NameError: name 'best_5_sites' is not defined